# Cheap-Talk Audit — PS1 notebook
COMSCI/ECON 206 · Muhan Chen (mc956) · Autumn 2026 Session 1

This notebook replays the Hugging Face Static Space in Python, checks the indifference threshold $p^\star=0.6$, and reports a 1,000-seed robustness check. **All outputs are synthetic.** They are not findings about human buyers or real LLM agents.

Shortest local run: from `cheap-talk-audit/`, `python -m unittest tests.test_cheap_talk -v`, then Run all here.

The interactive game is the same 12-round mechanism plus a classroom debrief. This notebook adds the $p^\star$ modification and the Monte Carlo, which the live game does not compute.

## Three connected questions
1. **Economics.** After the counterpart payoff table is public, when should the principal Trust rather than Verify?
2. **Computation.** Can a seeded replay make the benchmark, the promise–action gap, and similarity bias $\Delta$ checkable in both JavaScript and Python?
3. **Behavior.** Does a payoff-irrelevant SAME profile raise trust once alignment is held constant, or is noise / literal promise-following a better explanation?

Hypothesis to check first: $\mathrm{EU}(\text{Trust})=10p-4$ crosses the Verify payoff $+2$ at $p^\star=0.6$.

In [ ]:
import json, sys
from pathlib import Path
ROOT = Path.cwd()
if (ROOT / 'src' / 'cheap_talk.py').exists():
    sys.path.insert(0, str(ROOT / 'src'))
elif (ROOT.parent / 'src' / 'cheap_talk.py').exists():
    sys.path.insert(0, str(ROOT.parent / 'src'))
    ROOT = ROOT.parent
else:
    sys.path.insert(0, '/content')
from cheap_talk import (INDIFFERENCE_P, expected_trust, evaluate_seed,
                        indifference_check, monte_carlo, P_KEEP, PAY)
print('p* =', INDIFFERENCE_P)
print('EU(Trust | 0.85) =', expected_trust(0.85), '> Verify', PAY['verify'])
print('EU(Trust | 0.25) =', expected_trust(0.25), '< Verify', PAY['verify'])
print('EU(Trust | 0.60) =', expected_trust(0.60))

## Baseline: seed 206, four policies
Predict before running: the deployed Space reported benchmark $+48$, always-Verify $+24$, always-Trust $+22$, similarity $+18$, gap $4/11$.

In [ ]:
ev = evaluate_seed(206)
for name, rec in ev.items():
    print(f"{name:16} total={rec['total']:+d}  match={rec['benchmark_match']}/12  "
          f"Δ={rec['similarity_bias_pp']:+d}pp  gap={rec['gap_overall']}")
assert ev['benchmark']['total'] == 48
assert ev['similarity']['total'] == 18
print('seed 206 matches the deployed Space')

## One-assumption change (Wednesday protocol)
Change only the aligned keep-probability from 0.85 to $p^\star=0.6$. Prediction: Trust and Verify become equally good in expectation when aligned, so Trust-if-aligned need not remain uniquely best on the realized seed.

In [ ]:
print(indifference_check())
mod = evaluate_seed(206, p_keep={'aligned': INDIFFERENCE_P, 'misaligned': 0.25})
for name, rec in mod.items():
    print(f"{name:16} total={rec['total']:+d}  (benchmark on this modified seed {rec['benchmark_total']:+d})")
print('Interpretation: always-Verify (+24) now beats the old Trust-if-aligned rule (+18) on this seed.')
print('This does not say the original 0.85/0.25 model was wrong; it shows the unique recommendation depended on that one number.')

## Distribution check: 1,000 seeds
A single seed is a demonstration. Do mean payoffs preserve the claim that following resemblance is worse than never trusting?

In [ ]:
mc_path = ROOT / 'outputs' / 'ps1_validation.json'
if mc_path.exists():
    saved = json.loads(mc_path.read_text())['monte_carlo_1000']
    print('Loaded saved Monte Carlo from', mc_path)
    print(saved)
else:
    saved = monte_carlo(1000, start=1)
    print(saved)
assert saved['mean_totals']['similarity'] < saved['mean_totals']['always_verify']
print('Mean ranking supports: benchmark > always-Verify > similarity > always-Trust.')
print('The exact seed-206 order is not universal; see share_strict_seed206_ranking.')

## What this does and does not support
- Supports: the instrument is portable; $p^\star$ is the correct indifference cut; on average, a similarity policy loses to always-Verify under the stated placeholders.
- Does not support: claims about real people, real LLM agents, or classroom learning gains (SDG 4). Those remain planned tests.
- Competing behavioral explanations (noise vs resemblance vs literalism) are identified by the $2\times 2$ design, not by these four robot policies.

## Sources
Crawford and Sobel (1982); DeBruine (2002); Akata et al. (2023); Nash (1950); Harsanyi (1968). Keep-probabilities 0.85/0.25 are teaching placeholders, not estimates.